# 线性回归

In [ ]:
import random
import torch
from d2l import torch as d2l

## 数据集生成
使用线性模型参数$\mathbf{w} = [2, -3.4]^\top$、$b = 4.2$
和噪声项$\epsilon$生成数据集及其标签：

$$\mathbf{y}= \mathbf{X} \mathbf{w} + b + \mathbf\epsilon.$$

In [ ]:
def synthetic_data(w, b, num_examples):  #@save
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1)) # 这里reshape相当于转成列向量

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)
print('features:', features[0],'\nlabel:', labels[0])

d2l.set_figsize()
d2l.plt.scatter(features[:, (1)].detach().numpy(), labels.detach().numpy(), 1);

## 数据集读取
定义一个data_iter函数， 该函数接收批量大小、特征矩阵和标签向量作为输入，生成大小为batch_size的小批量。

每个小批量包含一组特征和标签

In [ ]:
def data_iter(batch_size, features, labels):
    num_examples = len(features) # N
    indices = list(range(num_examples)) # 创建一个索引列表 [0, 1, 2, ..., N-1]
    # 这些样本是随机读取的，没有特定的顺序
    random.shuffle(indices) # 随机打乱索引列表的顺序
    for i in range(0, num_examples, batch_size): # 循环遍历，步长为 batch_size
        batch_indices = torch.tensor(
            indices[i: min(i + batch_size, num_examples)]) # 切片当前批次的索引，并转为 PyTorch 张量
        yield features[batch_indices], labels[batch_indices]

以10个索引为例，indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

打乱索引后，indices = [3, 0, 9, 1, 5, 2, 8, 4, 7, 6]

假设 batch_size = 3
- 第 1 次循环 (i=0) 切片：indices[0:3] → [3, 0, 9]
- 第 2 次循环 (i=3) 切片：indices[3:6] → [1, 5, 2]
- 第 3 次循环 (i=6) 切片：indices[6:9] → [8, 4, 7]
- 第 4 次循环 (i=9) 切片：indices[9:10] → [6] (min 防止越界)


yield 的核心作用：暂停与记忆

普通函数使用 return 返回结果后会立即结束，局部变量会被销毁，而使用 yield 的函数会变成生成器函数

当代码执行到 yield 时，函数会暂停执行，并将 yield 后面的值返回给调用者

函数的当前状态（包括局部变量 i、indices、代码执行位置等）会被保存下来

当下一次请求数据时，函数会从上次暂停的地方继续执行，直到遇到下一个 yield 或函数结束

所以函数每调用一次就会返回一个batch_size大小的数据集

In [ ]:
batch_size = 10

for X, y in data_iter(batch_size, features, labels):
    print(X, '\n', y)
    break

## 初始化模型参数
从均值为0、标准差为0.01的正态分布中采样随机数来初始化权重， 并将偏置初始化为0

对于线性回归可以将权重全部初始化为0，但对于多层神经网络，不可以将权重全部初始化为0

In [ ]:
w = torch.normal(0, 0.01, size=(2,1), requires_grad=True)
# w = torch.zeros((2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

## 定义模型

In [ ]:
def linreg(X, w, b):  #@save
    """线性回归模型"""
    return torch.matmul(X, w) + b

## 定义损失函数

In [ ]:
def squared_loss(y_hat, y):  #@save
    """均方损失"""
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

## 梯度下降

In [ ]:
def sgd(params, lr, batch_size):  #@save
    """小批量随机梯度下降"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

## 训练

In [ ]:
lr = 0.03
num_epochs = 3
net = linreg
loss = squared_loss

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y)  # X和y的小批量损失
        # 因为l形状是(batch_size,1)，而不是一个标量。l中的所有元素被加到一起，
        # 并以此计算关于[w,b]的梯度
        l.sum().backward()
        sgd([w, b], lr, batch_size)  # 使用参数的梯度更新参数
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')
        print(f'w的估计误差: {true_w - w.reshape(true_w.shape)}')
        print(f'b的估计误差: {true_b - b}')

# 调库实现

In [ ]:
from torch.utils import data

## 数据集读取
这里的实现原理和上面的data_iter函数相同

In [ ]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)
next(iter(data_iter))

## 定义模型
全连接层在Linear类中定义。 值得注意的是，需要将两个参数传递到nn.Linear中
- 第一个指定输入特征形状，即features的维度，为2
- 第二个指定输出特征形状，输出特征形状为单个标量，即labels的维度，为1

In [ ]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

##  初始化模型参数

In [ ]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

## 定义损失函数
L2范数，默认是 reduction=‘mean’，和上面的一致，也可以使用求和损失 reduction=sum

In [ ]:
loss = nn.MSELoss()

## 定义优化算法
小批量随机梯度下降算法是一种优化神经网络的标准工具

实例化时需要指定优化网络和学习率

In [ ]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## 训练

In [ ]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

In [ ]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)